# Day 10：Attention 形状与 mask 可视化

对应 [`docs/day10.md`](../docs/day10.md) 的**任务 4**。

目的是**把形状变换可视化**，以后忘了随时回来看。

环境里没有 matplotlib，第 4 节用字符矩阵画热图
（`■` 可见、`·` 屏蔽；热图用 `·░▒▓█` 表示权重大小）。


In [ ]:
import math

import torch

from mini_transformer import MultiHeadAttention
from mini_transformer.attention import (
    build_causal_mask,
    build_padding_mask,
    combine_attention_masks,
)


torch.manual_seed(0)
print(f"PyTorch: {torch.__version__}")


def print_visibility(mask, title):
    is_visible = mask == 0
    print(title)
    print(f"shape = {tuple(mask.shape)}")
    for row in is_visible.tolist():
        print(" ".join("■" if allowed else "·" for allowed in row))
    print()


## 1. 形状流水线

从 `[B, S, D]` 回到 `[B, S, D]`。设 `B=2`、`S=4`、`D=12`、`H=3`，则 `Dh=4`。

```text
hidden_states [B, S, D]
  → Wq / Wk / Wv              [B, S, D]
  → view(B, S, H, Dh)         [B, S, H, Dh]
  → transpose(1, 2)           [B, H, S, Dh]
  → Q @ Kᵀ / √Dh              [B, H, Sq, Skv]
  → softmax → @ V             [B, H, Sq, Dh]
  → transpose + reshape       [B, S, D]
  → Wo                        [B, S, D]
```


In [ ]:
batch_size, sequence_length = 2, 4
hidden_size, num_heads = 12, 3
head_dim = hidden_size // num_heads

model = MultiHeadAttention(hidden_size=hidden_size, num_heads=num_heads)
hidden_states = torch.randn(batch_size, sequence_length, hidden_size)

query = model.q_proj(hidden_states)
key = model.k_proj(hidden_states)
value = model.v_proj(hidden_states)

query_viewed = query.view(batch_size, sequence_length, num_heads, head_dim)
key_viewed = key.view(batch_size, sequence_length, num_heads, head_dim)
value_viewed = value.view(batch_size, sequence_length, num_heads, head_dim)

query_heads = query_viewed.transpose(1, 2)
key_heads = key_viewed.transpose(1, 2)
value_heads = value_viewed.transpose(1, 2)

scores = query_heads @ key_heads.transpose(-2, -1)
scaled_scores = scores / math.sqrt(head_dim)
probabilities = scaled_scores.softmax(dim=-1)
attention_heads = probabilities @ value_heads

merged = attention_heads.transpose(1, 2)
reshaped = merged.reshape(batch_size, sequence_length, hidden_size)
output = model.out_proj(reshaped)

rows = [
    ("hidden_states", hidden_states.shape),
    ("q_proj / k_proj / v_proj", query.shape),
    ("view(B, S, H, Dh)", query_viewed.shape),
    ("transpose → [B, H, S, Dh]", query_heads.shape),
    ("scores = Q @ Kᵀ", scores.shape),
    ("softmax 沿 Skv", probabilities.shape),
    ("weighted V", attention_heads.shape),
    ("transpose 回来", merged.shape),
    ("reshape → [B, S, D]", reshaped.shape),
    ("out_proj", output.shape),
]

print(f"{'步骤':<28} {'形状'}")
print("-" * 46)
for name, shape in rows:
    print(f"{name:<28} {tuple(shape)}")

assert output.shape == hidden_states.shape
assert torch.allclose(output, model(hidden_states))


## 2. mask 长什么样

`S=4`。causal 挡住未来，padding 挡住 PAD，合成后任一来源屏蔽即屏蔽。

第二条序列的有效 token 只有前两个：`[1, 1, 0, 0]`。


In [ ]:
causal_mask = build_causal_mask(
    query_len=4,
    key_value_len=4,
    device=torch.device("cpu"),
    dtype=torch.float32,
)
padding_mask = build_padding_mask(
    torch.tensor(
        [
            [1, 1, 1, 0],
            [1, 1, 0, 0],
        ]
    ),
    dtype=torch.float32,
)
combined_mask = combine_attention_masks(causal_mask, padding_mask)

print_visibility(causal_mask, "causal [Sq, Skv] = [4, 4]")
print("padding [B, 1, 1, Skv] =", tuple(padding_mask.shape))
print("batch 0 PAD 在最后一位:", (padding_mask[0, 0, 0] == 0).tolist())
print("batch 1 PAD 在后两位:  ", (padding_mask[1, 0, 0] == 0).tolist())
print()
print(f"combined shape = {tuple(combined_mask.shape)}  （可广播到 [B, H, Sq, Skv]）")
print()
print_visibility(combined_mask[0, 0], "combined batch 0（因果 + 最后一位 PAD）")
print_visibility(combined_mask[1, 0], "combined batch 1（因果 + 后两位 PAD）")


## 3. `Sq != Skv`

对比 Prefill（`Sq = Skv = 8`）和 Decode（`Sq=1, Skv=8`）。

Decode 不是错误：有 KV Cache 时，Query 只有最新 1 个 token，Key/Value 是全部历史。
中间分数从 `S × S` 变成 `1 × S`，计算从平方变成线性。


In [ ]:
def attention_shapes(query_len, key_value_len, *, batch_size=2, num_heads=3, head_dim=8):
    query = torch.randn(batch_size, num_heads, query_len, head_dim)
    key = torch.randn(batch_size, num_heads, key_value_len, head_dim)
    value = torch.randn(batch_size, num_heads, key_value_len, head_dim)
    scores = query @ key.transpose(-2, -1)
    output = scores.softmax(dim=-1) @ value
    causal = build_causal_mask(
        query_len,
        key_value_len,
        device=query.device,
        dtype=query.dtype,
    )
    return {
        "Q": query.shape,
        "K / V": key.shape,
        "scores [B, H, Sq, Skv]": scores.shape,
        "output [B, H, Sq, Dh]": output.shape,
        "causal mask": causal.shape,
        "score 元素数": scores[0, 0].numel(),
    }


prefill = attention_shapes(8, 8)
decode = attention_shapes(1, 8)

print(f"{'张量':<24} {'Prefill Sq=Skv=8':<22} {'Decode Sq=1, Skv=8'}")
print("-" * 70)
for name in prefill:
    print(f"{name:<24} {str(tuple(prefill[name]) if name != 'score 元素数' else prefill[name]):<22} {tuple(decode[name]) if name != 'score 元素数' else decode[name]}")

print()
print_visibility(
    build_causal_mask(8, 8, device=torch.device("cpu"), dtype=torch.float32),
    "Prefill causal：下三角，8 × 8",
)
print_visibility(
    build_causal_mask(1, 8, device=torch.device("cpu"), dtype=torch.float32),
    "Decode causal：一行全可见，1 × 8",
)


## 4. 注意力权重热图

随机 Q/K，加上因果掩码，看第一个 head 的 `[S, S]` 概率。

下三角应该明显更亮：位置 `i` 的质量集中在 `0..i`，未来位置接近 0。


In [ ]:
sequence_length = 8
head_dim = 8
query = torch.randn(1, 1, sequence_length, head_dim)
key = torch.randn(1, 1, sequence_length, head_dim)
causal_mask = build_causal_mask(
    query_len=sequence_length,
    key_value_len=sequence_length,
    device=query.device,
    dtype=query.dtype,
)
scores = query @ key.transpose(-2, -1) / math.sqrt(head_dim)
probabilities = (scores + causal_mask).softmax(dim=-1)[0, 0]

print("softmax 概率（保留两位）：")
for row in probabilities.tolist():
    print(" ".join(f"{weight:5.2f}" for weight in row))

print()
print("可见性（■ 可见，· 被因果 mask 屏蔽）：")
for row in (causal_mask == 0).tolist():
    print(" ".join("■" if allowed else "·" for allowed in row))

print()
print("权重强度（·░▒▓█，未来位置应接近 ·）：")
levels = "·░▒▓█"
for row in probabilities.tolist():
    cells = []
    for weight in row:
        index = min(int(weight * (len(levels) - 1) * 2), len(levels) - 1)
        cells.append(levels[index])
    print(" ".join(cells))

print()
print("每一行概率和:", probabilities.sum(dim=-1).tolist())
assert torch.allclose(probabilities.sum(dim=-1), torch.ones(sequence_length))
assert torch.allclose(probabilities.triu(diagonal=1), torch.zeros(sequence_length, sequence_length))


完成标准：

- [x] 能从 `[B,S,D]` 逐步写出到 `[B,H,S,Dh]` 再还原
- [x] 看清 causal / padding / 合成三种 mask
- [x] 能对比 Prefill `S×S` 与 Decode `1×S` 的中间形状
- [x] 热图上能认出下三角

对应实现：`src/mini_transformer/attention.py`  
对应测试：`tests/test_attention.py`、`tests/test_mask.py`
